In [0]:
import pyspark
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_text = \
        spark.read.text('/Volumes/prodsqldwh/adls/nlp_files/pg2701_moby_dick.txt')

df_text.show(truncate=False, n= 5)

In [0]:
df_text.explain()

In [0]:
df_text_v2 = \
df_text.select(
        regexp_replace(col('value'), '[_/”/‘/“’/./,/:/;/—/?/-]', '').alias('value')
    )\
    .select(
        split(col('value'), ' ').alias('value')
    )\
    .select(
        explode(col('value')).alias('value')
    )\
    .where(
        col('value') != ''
    )

In [0]:
short_10chars = \
    udf(
        lambda x: 
        True if not x or len(x) < 10 else False, BooleanType()
    )

df_text_v3 = \
    df_text_v2.select(
        col('value').alias('word'),
        short_10chars(col('word')).alias('less_10_chars')
    )\
    .show(truncate=False, n= 5)

In [0]:
df_text_v4 = \
    df_text_v2.withColumn(
        'id', monotonically_increasing_id()
    )\
    .select(
        col('value').alias('w1'),
        lead(col('value'), 1).over(Window.orderBy(col('id').asc())).alias('w2'),
        lead(col('value'), 2).over(Window.orderBy(col('id').asc())).alias('w3'),
        lead(col('value'), 3).over(Window.orderBy(col('id').asc())).alias('w4'),
        lead(col('value'), 4).over(Window.orderBy(col('id').asc())).alias('w5')
    )\
    .select(
        array(
            col('w1'),
            col('w2'),
            col('w3'),
            col('w4'),
            col('w5')
        ).alias('array_words'),
        element_at(col('array_words'), -1).alias('last_word')
    )\
    .filter(
        forall(col('array_words'), lambda x: x.isNotNull())
    )\
    # .show(truncate=False)

In [0]:
from pyspark.ml.feature import CountVectorizer

# Fitting CountVectorizer: 
cv = \
    CountVectorizer(
        inputCol='array_words', 
        outputCol='features'
    )
model = cv.fit(df_text_v4)
df_text_v5 = \
    model.transform(df_text_v4)
df_text_v5.show(truncate=False, n=5)

In [0]:
df_success = \
    df_text_v5.where(
        col('last_word').isin('Moby', 'Dick', 'Whale')
    )\
    .withColumn('label', lit(1))

df_not_success = \
    df_text_v5.where(
        ~col('last_word').isin('Moby', 'Dick', 'Whale')
    )\
    .withColumn('label', lit(0))\
    .limit(df_success.count())

# The data for the model must have the same number of success and not success rows
df_data_for_model = \
    df_success.union(df_not_success)

df_data_for_model.show(truncate=False, n=5)

In [0]:
from pyspark.ml.classification import LogisticRegression

df_train, df_test = df_data_for_model.randomSplit([0.8, 0.2], seed=42)

logistic = LogisticRegression(
    maxIter=20, 
    regParam=0.3, 
    elasticNetParam=0.0
)

df_fitted = logistic.fit(df_train)
df_fitted.summary.totalIterations

In [0]:
df_predicted = df_fitted.transform(df_test)
df_predicted\
    .select(
        col('array_words'),
        col('last_word'),
        col('label'),
        col('probability'),
        col('prediction')
    )\
    .show(truncate=False)

In [0]:
# Evaluating the model: 
df_eval = \
    df_fitted.evaluate(df_test)

print('AUC: %.5f' % df_eval.areaUnderROC)
print('Acuracy: %.5f' % df_eval.accuracy)
# The model has a success rate of 99.99